# 🎙️ 長時間会議 自動文字起こし・議事録作成システム

## 使い方
1. `MeetingTranscript/01_input` フォルダに音声ファイルを入れる
2. スプレッドシートの「本日の参加者」を更新する
3. **「すべてのセルを実行」を押して待つだけ** ☕

---

## Step 1: 環境セットアップ
必要なライブラリをインストールします（初回のみ数分かかります）

In [5]:
# ライブラリのインストール18:1318:13
# ※ 初回実行時は数分かかります
!pip install -q whisperx
!pip install -q gspread google-auth
!pip install -q google-generativeai

print('✅ ライブラリのインストール完了')
print('⚠️ ランタイムを再起動してから次のセルに進んでください')
print('   → メニュー「ランタイム」→「ランタイムを再起動」')

✅ ライブラリのインストール完了
⚠️ ランタイムを再起動してから次のセルに進んでください
   → メニュー「ランタイム」→「ランタイムを再起動」


In [6]:
# Google Drive をマウント
from google.colab import drive
drive.mount('/content/drive')

print('✅ Google Drive のマウント完了')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive のマウント完了


In [7]:
# APIキーの読み込み（Colab のシークレット機能を使用）
# ※ 左サイドバー 🔑 アイコン → シークレットに以下を登録してください
#   - HF_TOKEN            : Hugging Face のトークン
#   - GEMINI_API_KEY      : Gemini API キー
#   - NOTION_TOKEN        : Notion インテグレーションのトークン
#   - NOTION_DATABASE_ID  : Notion データベースの ID

from google.colab import userdata

HF_TOKEN           = userdata.get('HF_TOKEN')
GEMINI_API_KEY     = userdata.get('GEMINI_API_KEY')
NOTION_TOKEN       = userdata.get('NOTION_TOKEN')
NOTION_DATABASE_ID = userdata.get('NOTION_DATABASE_ID')

# 登録漏れがないか確認
missing = [name for name, val in [
    ('HF_TOKEN', HF_TOKEN),
    ('GEMINI_API_KEY', GEMINI_API_KEY),
    ('NOTION_TOKEN', NOTION_TOKEN),
    ('NOTION_DATABASE_ID', NOTION_DATABASE_ID),
] if not val]

if missing:
    raise ValueError(f'❌ 以下のシークレットが未登録です: {missing}')

print('✅ APIキーの読み込み完了')

✅ APIキーの読み込み完了


In [8]:
# ========================================
# ⚙️ 設定（ここだけ自分の環境に合わせて変更）
# ========================================

# Google Drive 上のフォルダパス
BASE_DIR        = '/content/drive/MyDrive/MeetingTranscript'
INPUT_AUDIO_DIR = f'{BASE_DIR}/01_input'
DONE_AUDIO_DIR  = f'{BASE_DIR}/02_processed'
TEXT_OUTPUT_DIR = f'{BASE_DIR}/03_output'

# スプレッドシートの設定
SPREADSHEET_NAME = 'MeetingTranscript'  # スプレッドシートの名前（04_management フォルダ内）
WORKSHEET_NAME   = '本日の参加者'        # シート名

# Gemini の設定
GEMINI_MODEL         = 'gemini-2.5-flash-lite'  # 無料枠あり（RPD: 20）
SPEAKER_SAMPLE_CHARS = 3000   # 話者特定に使う冒頭文字数
CHUNK_SIZE           = 10000  # 整形処理の分割単位（文字数）

print('✅ 設定の読み込み完了')
print(f'   入力フォルダ : {INPUT_AUDIO_DIR}')
print(f'   出力フォルダ : {TEXT_OUTPUT_DIR}')

✅ 設定の読み込み完了
   入力フォルダ : /content/drive/MyDrive/MeetingTranscript/01_input
   出力フォルダ : /content/drive/MyDrive/MeetingTranscript/03_output


---
## Step 2: 音声ファイルの文字起こし（WhisperX）
GPUを使って音声を文字起こし＋話者分離します

In [9]:
import os
import glob

# 入力フォルダから音声ファイルを1本取得
audio_files = glob.glob(f'{INPUT_AUDIO_DIR}/*.mp3') + \
              glob.glob(f'{INPUT_AUDIO_DIR}/*.m4a') + \
              glob.glob(f'{INPUT_AUDIO_DIR}/*.wav')

if not audio_files:
    raise FileNotFoundError(f'❌ {INPUT_AUDIO_DIR} に音声ファイルがありません')

# 最初の1ファイルを処理対象にする
audio_path = audio_files[0]
print(f'✅ 処理対象ファイル: {os.path.basename(audio_path)}')

✅ 処理対象ファイル: 新規録音 6.m4a


In [10]:
import whisperx

# デバイスの設定（GPUが使える場合はcuda、使えない場合はcpu）
device = 'cuda'
compute_type = 'float16'

print('🎙️ WhisperX で文字起こし中... (しばらくかかります)')

# モデルの読み込み（largeモデルで高精度）
model = whisperx.load_model('large-v3', device, compute_type=compute_type)

# 文字起こし実行
audio = whisperx.load_audio(audio_path)
result = model.transcribe(audio, batch_size=16, language='ja')

print(f'✅ 文字起こし完了（{len(result["segments"])} セグメント）')

🎙️ WhisperX で文字起こし中... (しばらくかかります)
2026-04-18 09:43:39 - whisperx.asr - INFO - No language specified, language will be detected for each audio file (increases inference time)
2026-04-18 09:43:39 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


INFO: Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.1. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`
INFO:lightning.pytorch.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.1. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`
/usr/local/lib/python3.12/dist-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issue

✅ 文字起こし完了（71 セグメント）


In [11]:
# 話者分離（誰が話しているか識別）
print('👥 話者分離中...')

# 音素アライメント
model_a, metadata = whisperx.load_align_model(language_code='ja', device=device)
result = whisperx.align(result['segments'], model_a, metadata, audio, device)

# 話者ラベルの付与（whisperx 3.x系の書き方）
from whisperx.diarize import DiarizationPipeline
diarize_model = DiarizationPipeline(token=HF_TOKEN, device=device)
diarize_segments = diarize_model(audio)
result = whisperx.assign_word_speakers(diarize_segments, result)

print('✅ 話者分離完了')

👥 話者分離中...


/usr/local/lib/python3.12/dist-packages/transformers/configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


2026-04-18 09:45:51 - whisperx.diarize - INFO - Loading diarization model: pyannote/speaker-diarization-community-1


/usr/local/lib/python3.12/dist-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1839.)
  std = sequences.std(dim=-1, correction=1)


✅ 話者分離完了


In [12]:
# 文字起こし結果を「SPEAKER_XX: テキスト」形式に整形
transcript_lines = []
for segment in result['segments']:
    speaker = segment.get('speaker', 'SPEAKER_UNKNOWN')
    text    = segment['text'].strip()
    if text:  # 空行はスキップ
        transcript_lines.append(f'[{speaker}] {text}')

# テキスト全文を結合
full_transcript = '\n'.join(transcript_lines)

# 一時保存（途中でColabが落ちても安心）
os.makedirs(TEXT_OUTPUT_DIR, exist_ok=True)
raw_text_path = f'{TEXT_OUTPUT_DIR}/raw_transcript.txt'
with open(raw_text_path, 'w', encoding='utf-8') as f:
    f.write(full_transcript)

print(f'✅ 生テキストを保存: {raw_text_path}')
print(f'   総文字数: {len(full_transcript)} 文字')
print(f'   冒頭サンプル:')
print(full_transcript[:300])

✅ 生テキストを保存: /content/drive/MyDrive/MeetingTranscript/03_output/raw_transcript.txt
   総文字数: 9079 文字
   冒頭サンプル:
[SPEAKER_00] そうですね、まあまあまあ、はいそうですね、そうですよね、で、今家賃が6万5千7百円でしたっけ?えっと、ちょっと新しいですかね?えっと、今ってすいません、こっちかな?40…何の方でしたっけ?すいません404号室ですね
[SPEAKER_00] 4万4千円すみません6万7千円ですかねそうですねいただいた数字を正しいものとして前提で考えた場合なんですけど例えば建物の固定資産課税標準額が1800万ぐらいあってそれを建物の床面積で割ってで
[SPEAKER_00] 石井さんが借りている部分45.38平米の分を出すと固定資産税課税標準額が89万ぐらいになるんですねざっくり言うと土


---
## Step 3: スプレッドシートから参加者情報を取得

In [13]:
import gspread
from google.colab import auth
from google.auth import default

# Colab の Google アカウントで認証
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# スプレッドシートを開く
spreadsheet = gc.open(SPREADSHEET_NAME)
worksheet   = spreadsheet.worksheet(WORKSHEET_NAME)

# 参加者データを取得（ヘッダー行を除く）
records = worksheet.get_all_records()

if not records:
    raise ValueError('❌ スプレッドシートに参加者情報がありません。更新してください。')

# 参加者情報を確認
print('✅ 参加者情報を取得しました:')
for row in records:
    print(f"   {row['名前']}（{row['役割']}）: {row['プロフィール']}")

✅ 参加者情報を取得しました:
   石井（経営者）: 経営者
   宮本（税理士）: 石井の担当税理士


---
## Step 4: 話者特定（Gemini）→ 名前置換（Python）→ テキスト整形

In [14]:
import google.generativeai as genai
import json
import re

# Gemini の初期化
genai.configure(api_key=GEMINI_API_KEY)
gemini = genai.GenerativeModel(GEMINI_MODEL)

# --- 話者特定 ---
# 参加者プロフィールをテキスト化
profile_text = '\n'.join([
    f"- {row['名前']}（{row['役割']}）: {row['プロフィール']}"
    for row in records
])

# 冒頭3000文字を抽出
sample_text = full_transcript[:SPEAKER_SAMPLE_CHARS]

# Gemini に話者特定を依頼
identify_prompt = f"""以下の会議の文字起こし（冒頭部分）と参加者情報を元に、
SPEAKER_XX が誰に対応するか特定してください。

【参加者情報】
{profile_text}

【文字起こし（冒頭）】
{sample_text}

必ず以下のJSON形式のみで回答してください。余分なテキストは不要です。
{{"SPEAKER_00": "名前", "SPEAKER_01": "名前", ...}}
"""

print('🤖 Gemini で話者を特定中...')
response = gemini.generate_content(identify_prompt)

# JSON部分だけ抽出してパース
json_match = re.search(r'\{.*\}', response.text, re.DOTALL)
if not json_match:
    raise ValueError(f'❌ Gemini のレスポンスからJSONを取得できませんでした:\n{response.text}')

speaker_map = json.loads(json_match.group())
print('✅ 話者の対応表:')
for speaker_id, name in speaker_map.items():
    print(f'   {speaker_id} → {name}')

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


🤖 Gemini で話者を特定中...
✅ 話者の対応表:
   SPEAKER_00 → 宮本（税理士）
   SPEAKER_01 → 石井（経営者）


In [15]:
# --- Python で機械的に名前を置換 ---
replaced_transcript = full_transcript
for speaker_id, name in speaker_map.items():
    replaced_transcript = replaced_transcript.replace(f'[{speaker_id}]', f'{name}:')

# マッピングされなかった SPEAKER_XX が残っていないか確認
remaining = re.findall(r'\[SPEAKER_\d+\]', replaced_transcript)
if remaining:
    print(f'⚠️ 未置換の話者が残っています: {set(remaining)}')
    print('   スプレッドシートの参加者情報を確認してください')
else:
    print('✅ 全話者の名前置換が完了しました')

✅ 全話者の名前置換が完了しました


In [16]:
# --- Gemini で専門用語補正＋対話形式に整形 ---
# テキストを1万文字ずつ分割して処理
def split_text(text, chunk_size):
    """テキストを改行単位で chunk_size 文字以内に分割する"""
    chunks = []
    current_chunk = []
    current_len = 0

    for line in text.split('\n'):
        line_len = len(line) + 1  # 改行分を加算
        if current_len + line_len > chunk_size and current_chunk:
            chunks.append('\n'.join(current_chunk))
            current_chunk = []
            current_len = 0
        current_chunk.append(line)
        current_len += line_len

    if current_chunk:
        chunks.append('\n'.join(current_chunk))

    return chunks

chunks = split_text(replaced_transcript, CHUNK_SIZE)
print(f'✅ テキストを {len(chunks)} チャンクに分割しました')

# 各チャンクを Gemini で整形
formatted_chunks = []
for i, chunk in enumerate(chunks):
    print(f'🤖 チャンク {i+1}/{len(chunks)} を整形中...')

    format_prompt = f"""以下は会議の文字起こしです。
専門用語の明らかな誤認識を補正し、読みやすい対話形式に整えてください。

ルール:
- 話者名と発言内容はそのまま保持する（例: 田中: 〜〜〜）
- 意味が変わるような要約・削除はしない（全文を保持）
- 明らかな誤字・誤変換のみ修正する

【文字起こし】
{chunk}
"""

    response = gemini.generate_content(format_prompt)
    formatted_chunks.append(response.text)

# 全チャンクを結合
final_text = '\n\n'.join(formatted_chunks)
print(f'✅ テキスト整形完了（総文字数: {len(final_text)} 文字）')

✅ テキストを 1 チャンクに分割しました
🤖 チャンク 1/1 を整形中...
✅ テキスト整形完了（総文字数: 9622 文字）


---
## Step 5: Notion に議事録ページを作成

In [26]:
import requests
from datetime import date

# Notion API の共通設定
NOTION_API_VERSION = '2022-06-28'
NOTION_HEADERS = {
    'Authorization': f'Bearer {NOTION_TOKEN}',
    'Notion-Version': NOTION_API_VERSION,
    'Content-Type': 'application/json',
}

def notion_post(url, body):
    """Notion API に POST リクエストを送る（ページ作成用）"""
    res = requests.post(url, headers=NOTION_HEADERS, json=body)
    if not res.ok:
        raise RuntimeError(f'Notion API エラー {res.status_code}: {res.text}')
    return res.json()

def notion_patch(url, body):
    """Notion API に PATCH リクエストを送る（ブロック追記用）"""
    res = requests.patch(url, headers=NOTION_HEADERS, json=body)
    if not res.ok:
        raise RuntimeError(f'Notion API エラー {res.status_code}: {res.text}')
    return res.json()

# データベースIDをUUID形式に正規化
db_id_raw = NOTION_DATABASE_ID.replace('-', '').strip()
db_id = f'{db_id_raw[0:8]}-{db_id_raw[8:12]}-{db_id_raw[12:16]}-{db_id_raw[16:20]}-{db_id_raw[20:32]}'

# 本日の日付とページタイトルを生成
today = date.today().strftime('%Y-%m-%d')
page_title = f'議事録_{today}'

# 参加者リストを文字列化
participants = '、'.join([
    f"{row['名前']}（{row['役割']}）" for row in records
])

# Notion のブロック形式に変換
def text_to_blocks(text):
    """テキストを Notion の段落ブロックのリストに変換する"""
    blocks = []
    for line in text.split('\n'):
        blocks.append({
            'object': 'block',
            'type': 'paragraph',
            'paragraph': {
                'rich_text': [{'type': 'text', 'text': {'content': line}}]
            }
        })
    return blocks

print(f'📝 Notion にページを作成中: {page_title}')

# ページ作成（POST）
new_page = notion_post(
    'https://api.notion.com/v1/pages',
    {
        'parent': {'database_id': db_id},
        'properties': {
            '名前': {
                'title': [{'text': {'content': page_title}}]
            }
        }
    }
)

page_id = new_page['id']
page_url = new_page['url']
print(f'✅ ページ作成完了: {page_url}')

# 本文の冒頭に日付・参加者情報を追加（PATCH）
header_blocks = [
    {
        'object': 'block',
        'type': 'paragraph',
        'paragraph': {'rich_text': [{'type': 'text', 'text': {'content': f'日付: {today}'}}]}
    },
    {
        'object': 'block',
        'type': 'paragraph',
        'paragraph': {'rich_text': [{'type': 'text', 'text': {'content': f'参加者: {participants}'}}]}
    },
    {
        'object': 'block',
        'type': 'divider',
        'divider': {}
    }
]
notion_patch(f'https://api.notion.com/v1/blocks/{page_id}/children', {'children': header_blocks})

# 本文を100ブロックずつ分けて追加（PATCH・Notion API の制限）
all_blocks = text_to_blocks(final_text)
for i in range(0, len(all_blocks), 100):
    batch = all_blocks[i:i+100]
    notion_patch(f'https://api.notion.com/v1/blocks/{page_id}/children', {'children': batch})
    print(f'   ブロック追加中... {min(i+100, len(all_blocks))}/{len(all_blocks)}')

print(f'✅ Notion ページの作成完了！')
print(f'   🔗 {page_url}')

📝 Notion にページを作成中: 議事録_2026-04-18
✅ ページ作成完了: https://www.notion.so/_2026-04-18-3466f8be022781fc99f2c8d632bd702d
   ブロック追加中... 100/129
   ブロック追加中... 129/129
✅ Notion ページの作成完了！
   🔗 https://www.notion.so/_2026-04-18-3466f8be022781fc99f2c8d632bd702d


---
## Step 6: 後片付け

In [27]:
import shutil
from datetime import datetime

# 処理済み音声ファイルを移動
os.makedirs(DONE_AUDIO_DIR, exist_ok=True)
done_path = os.path.join(DONE_AUDIO_DIR, os.path.basename(audio_path))
shutil.move(audio_path, done_path)
print(f'✅ 音声ファイルを移動: {done_path}')

# 最終テキストを保存
final_text_path = f'{TEXT_OUTPUT_DIR}/minutes_{today}.md'
with open(final_text_path, 'w', encoding='utf-8') as f:
    f.write(f'# {page_title}\n\n')
    f.write(f'参加者: {participants}\n\n')
    f.write('---\n\n')
    f.write(final_text)
print(f'✅ 議事録を保存: {final_text_path}')

# 処理ログをスプレッドシートに記録
try:
    log_sheet = spreadsheet.worksheet('処理ログ')
except gspread.WorksheetNotFound:
    log_sheet = spreadsheet.add_worksheet(title='処理ログ', rows=1000, cols=5)
    log_sheet.append_row(['日時', 'ファイル名', '参加者', 'Notionページ'])

log_sheet.append_row([
    datetime.now().strftime('%Y-%m-%d %H:%M'),
    os.path.basename(done_path),
    participants,
    page_url
])
print('✅ 処理ログを記録しました')

print()
print('=' * 50)
print('🎉 すべての処理が完了しました！')
print(f'   Notion ページ: {page_url}')
print('=' * 50)

✅ 音声ファイルを移動: /content/drive/MyDrive/MeetingTranscript/02_processed/新規録音 6.m4a
✅ 議事録を保存: /content/drive/MyDrive/MeetingTranscript/03_output/minutes_2026-04-18.md
✅ 処理ログを記録しました

🎉 すべての処理が完了しました！
   Notion ページ: https://www.notion.so/_2026-04-18-3466f8be022781fc99f2c8d632bd702d
